In [22]:
import math
import re
from datetime import datetime
import jieba
import pandas as pd
import requests

# 設定 Pandas 顯示寬度，方便在 Jupyter 看完整標題
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 10)
print("✅ Cell 1: 套件與環境初始化完成")

✅ Cell 1: 套件與環境初始化完成


# kalshi

In [23]:
# ==========================================
# 📈 Cell 2：Kalshi 爬蟲模組（過濾遠期合約 + 鎖定近期真實流動性）
# ==========================================


def get_real_kalshi_markets(limit=100, max_days_ahead=365):
  url = "https://api.elections.kalshi.com/trade-api/v2/events"
  params = {"limit": limit, "status": "open", "with_nested_markets": "true"}
  headers = {"accept": "application/json"}

  columns = [
      "ticker",
      "event_title",
      "market_title",
      "category",
      "implied_prob",
      "volume",
      "open_interest",
      "close_date",
  ]

  try:
    resp = requests.get(url, headers=headers, params=params, timeout=12)
    if resp.status_code != 200:
      print(f"❌ Kalshi 請求失敗: {resp.status_code}")
      return pd.DataFrame(columns=columns)

    events = resp.json().get("events", [])
    cleaned = []
    now_utc = datetime.now().astimezone()

    for ev in events:
      ev_title = ev.get("title", "")
      category = ev.get("category", "General")
      markets = ev.get("markets", [])

      # 快速排除明顯是 2028~2040 的超遠期事件標題
      if any(str(yr) in ev_title for yr in range(2028, 2040)):
        continue

      for m in markets:
        # 1. 檢查到期時間，排除超過 1 年以上的超遠期合約
        close_str = m.get("close_time") or m.get("expiration_time")
        close_date_fmt = "N/A"
        if close_str:
          try:
            close_dt = datetime.fromisoformat(close_str.replace("Z", "+00:00"))
            days_diff = (close_dt - now_utc).days
            if days_diff > max_days_ahead or days_diff < 0:
              continue
            close_date_fmt = close_dt.strftime("%Y-%m-%d")
          except Exception:
            pass

        # 2. 取得成交量與未平倉量
        vol = float(
            m.get("volume_fp") or m.get("volume") or m.get("volume_24h") or 0
        )
        oi = float(m.get("open_interest_fp") or m.get("open_interest") or 0)

        # 3. 取得最新報價並轉為機率
        price_raw = (
            m.get("last_price_dollars")
            or m.get("yes_bid_dollars")
            or m.get("yes_ask_dollars")
            or m.get("last_price")
            or m.get("yes_bid")
            or m.get("yes_ask")
        )

        if price_raw is not None:
          try:
            p_val = float(price_raw)
          except ValueError:
            continue
          if p_val <= 0:
            continue
          implied_prob = (
              round(p_val / 100.0, 4) if p_val > 1.0 else round(p_val, 4)
          )
        else:
          continue

        cleaned.append({
            "ticker": m.get("ticker", ""),
            "event_title": ev_title,
            "market_title": (
                m.get("title") or m.get("yes_sub_title") or ev_title
            ),
            "category": category,
            "implied_prob": implied_prob,
            "volume": vol,
            "open_interest": oi,
            "close_date": close_date_fmt,
        })

    df = pd.DataFrame(cleaned, columns=columns)
    if not df.empty:
      df = df.sort_values(by="volume", ascending=False).reset_index(drop=True)
    return df

  except Exception as e:
    print(f"❌ Kalshi 爬蟲異常: {e}")
    return pd.DataFrame(columns=columns)


# 執行並預覽（只抓未來 365 天內結算的近期合約）
df_kalshi = get_real_kalshi_markets(limit=100, max_days_ahead=365)
print(f"✅ Cell 2: 成功抓到 {len(df_kalshi)} 筆【近期】有效 Kalshi 合約")
display(df_kalshi.head(10))

✅ Cell 2: 成功抓到 11 筆【近期】有效 Kalshi 合約


,ticker,event_title,market_title,category,implied_prob,volume,open_interest,close_date
0,KXMLBDEBUT-RCHOLOWSKY-26NOV01,Roch Cholowsky: Debut Date,"Will Roch Cholowsky play in a game for any team in the MLB before Nov 1, 2026?",Sports,0.04,19677.82,11637.84,2026-11-01
1,KXRIEMANNRES-40-27JAN01,When will the Riemann Hypothesis be resolved?,Will the Riemann Hypothesis be resolved before 2027?,Science and Technology,0.10,2811.95,2266.29,2027-01-07
2,KXHODGE-32-27JAN01,When will the Hodge Conjecture be resolved?,Will the Hodge Conjecture be resolved before 2027?,Science and Technology,0.11,2513.72,1789.20,2027-01-07
3,KXNFLRETIRE-KCOUSINS8-2728,Kirk Cousins: Retirement,Will Kirk Cousins announce his retirement before the 2027-28 NFL season?,Sports,0.88,1541.22,774.22,2027-09-30
4,KXFDAAPPROVE-MDMA-27JAN01,Will the FDA approve MDMA for PTSD?,"Will the FDA approve midomafetamine / MDMA for PTSD before Jan 1, 2027?",Science and Technology,0.06,1286.02,671.61,2027-01-01
5,KXBIRCHSWINN-32-27JAN01,When will Birch and Swinnerton-Dyer be resolved?,Will Birch and Swinnerton-Dyer be resolved before 2027?,Science and Technology,0.07,1249.94,980.00,2027-01-07
6,KXNFLRETIRE-MEVANS5-2728,Mike Evans: Retirement,Will Mike Evans announce his retirement before the 2027-28 NFL season?,Sports,0.80,758.64,164.34,2027-09-30
7,KXNFLRETIRE-MSTAFFORD9-2728,Matthew Stafford: Retirement,Will Matthew Stafford announce his retirement before the 2027-28 NFL season?,Sports,0.40,497.83,217.37,2027-09-30
8,KXPNP-32-27JAN01,When will P vs NP be resolved?,Will P vs NP be resolved before 2027?,Science and Technology,0.97,223.66,210.08,2027-01-07
9,KXYANGMILLS-32-27JAN01,When will Yang–Mills and the Mass Gap be resolved?,Will Yang–Mills and the Mass Gap be resolved before 2027?,Science and Technology,0.11,29.69,24.69,2027-01-07


In [24]:
import os
from datetime import datetime

# 定義儲存路徑與建立資料夾
base_path = r"C:\Users\JDA\Desktop\kalshi 台灣股市預測\kalshi_data"
os.makedirs(base_path, exist_ok=True)

# 執行存檔 (檔名加上年月日時分)
if 'df_kalshi' in locals() and not df_kalshi.empty:
    file_name = f"{base_path}\\kalshi_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df_kalshi.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"📁 Kalshi 宏觀預測已獨立儲存至: {file_name}")

📁 Kalshi 宏觀預測已獨立儲存至: C:\Users\JDA\Desktop\kalshi 台灣股市預測\kalshi_data\kalshi_20261027_1644.csv


# 測試

In [25]:
# ==========================================
# 🔍 Cell 1: 掃描並過濾可用模型清單
# ==========================================
import google.generativeai as genai

import os
from dotenv import load_dotenv
load_dotenv()  # 從同資料夾的 .env 讀取金鑰
API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=API_KEY)

print("開始掃描此金鑰支援文字生成的可用模型...\n")
try:
    # 篩選出具備 generateContent 能力的模型
    available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
    
    if not available_models:
        print("⚠️ 警告：金鑰有效，但此帳號未開啟任何文字生成模型權限。")
    else:
        print(f"✅ 金鑰驗證通過！共找到 {len(available_models)} 個可用模型。")
        
        # 優先篩選出目前最穩定且無限制的 1.5 系列版本
        print("\n強烈建議使用的穩定版模型 (請複製這些名稱)：")
        recommended = [m for m in available_models if "1.5" in m and ("flash" in m or "pro" in m)]
        for m in recommended:
            print(f"  👉 {m.replace('models/', '')}")
            
        print("\n完整可用清單前 100 筆：")
        for m in available_models[:100]:
            print(f"  - {m}")

except Exception as e:
    print(f"❌ 掃描失敗，網路阻擋或金鑰格式異常：{e}")

開始掃描此金鑰支援文字生成的可用模型...

✅ 金鑰驗證通過！共找到 46 個可用模型。

強烈建議使用的穩定版模型 (請複製這些名稱)：

完整可用清單前 100 筆：
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/gemini-3-pro-image
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.1-flash-image
  - models/gemini-3.1-flash-lite-image
  - models/gemini-nano-banana-2.1
  - models/gemini-3.5-flash
  - models/gemini-3.5-flash-lite
  - models/gemini-omni-flash-preview
  - models/gemini-omni-1

In [26]:
# ==========================================
# 🏥 Cell 2: 全方位 API 狀態與錯誤深度診斷
# ==========================================
import google.generativeai as genai
from google.api_core import exceptions

import os
from dotenv import load_dotenv
load_dotenv()  # 從同資料夾的 .env 讀取金鑰
API_KEY = os.getenv("GEMINI_API_KEY")

# 強制指定一個最普遍支援的 1.5 版模型進行真實通訊測試
# (若 Cell 1 建議其他名稱，可在此替換)
TEST_MODEL = "gemini-3.5-flash" 

genai.configure(api_key=API_KEY)
print(f"啟動深度診斷，測試標的：{TEST_MODEL}\n")

try:
    print("⏳ 正在發送測試封包至 Google API 計費閘道...")
    model = genai.GenerativeModel(TEST_MODEL)
    
    # 發送極短 prompt 以測試實際通訊與權限
    response = model.generate_content("這是一次診斷測試，請回覆『連線正常』四個字。")
    
    print(f"✅ 【診斷通過】API 狀態完美！")
    print(f"   模型實際回覆：{response.text.strip()}")
    print("   結論：金鑰正確、配額充足、未被限速，且模型名稱無誤。程式碼可直接上線抓取新聞！")

except exceptions.InvalidArgument as e:
    print("❌ 【錯誤 400：金鑰格式錯誤】")
    print("   診斷：API Key 字串不完整、多出空白，或包含了不必要的引號。")
    print(f"   系統原錯：{e}")

except exceptions.PermissionDenied as e:
    print("❌ 【錯誤 403：權限遭拒 / 金鑰失效】")
    print("   診斷：這把金鑰已被 Google 停用，或該 GCP 專案的 Generative Language API 服務被關閉。")
    print(f"   系統原錯：{e}")

except exceptions.NotFound as e:
    print("❌ 【錯誤 404：找不到模型】")
    print(f"   診斷：Google 伺服器拒絕了 `{TEST_MODEL}`。這通常是因為你使用的帳號無權存取該特定版本。")
    print("   解法：請回到 Cell 1 挑選清單中有出現的模型名稱替換。")
    print(f"   系統原錯：{e}")

except exceptions.ResourceExhausted as e:
    print("❌ 【錯誤 429：配額耗盡或計費攔截】")
    print("   診斷：你的請求被 Google 計費系統擋下，絕對是以下兩種情況之一：")
    print("   1. [全新專案] 該 GCP 專案未綁定信用卡 (Billing)。目前 Google 經常直接攔截未綁卡專案的首筆請求。")
    print("   2. [已用專案] 你今天的 1,500 次免費呼叫額度 (Quota) 已經徹底歸零。")
    print(f"   系統原錯：{e}")

except Exception as e:
    print(f"❌ 【未知異常：{type(e).__name__}】")
    print("   診斷：非 Google API 限制。可能是本機網路環境被防火牆阻擋，或 requests 模組缺少憑證。")
    print(f"   系統原錯：{e}")

啟動深度診斷，測試標的：gemini-3.5-flash

⏳ 正在發送測試封包至 Google API 計費閘道...
❌ 【錯誤 429：配額耗盡或計費攔截】
   診斷：你的請求被 Google 計費系統擋下，絕對是以下兩種情況之一：
   1. [全新專案] 該 GCP 專案未綁定信用卡 (Billing)。目前 Google 經常直接攔截未綁卡專案的首筆請求。
   2. [已用專案] 你今天的 1,500 次免費呼叫額度 (Quota) 已經徹底歸零。
   系統原錯：429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.
Please retry in 22h15m32.940229706s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  description: "Generate content requests per day limit exceeded on Fre

# 台灣新聞爬蟲+語意分析

In [27]:
# ==========================================
# 📰 Cell 3 (終極 NLP 版)：自動化 Jieba 字典 + 暴力爬蟲 + 放寬 AI 逾時
# ==========================================

import google.generativeai as genai
import requests
import pandas as pd
import json
import time
import jieba
import jieba.analyse
from datetime import datetime
from bs4 import BeautifulSoup

# 1. API 初始化
import os
from dotenv import load_dotenv
load_dotenv()  # 從同資料夾的 .env 讀取金鑰
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-3.5-flash')

# 2. 定義台股白名單
VALID_STOCK_DICT = {
    "2330": "台積電", "2317": "鴻海", "2454": "聯發科", 
    "2308": "台達電", "2881": "富邦金", "3023": "信邦",
    "6672": "騰輝電子", "3376": "新日興", "1503": "士電",
    "1513": "中興電", "1514": "亞力", "1519": "華城",
    "6669": "緯穎", "3406": "玉晶光", "3163": "波若威"
}

# ---------------------------------------------------------
# 💡 核心優化 A：動態注入 Jieba 字典 (財經詞彙 + 台股公司名)
# ---------------------------------------------------------
custom_finance_words = [
    "法說會", "先進製程", "CoWoS", "輝達", "外資", "投信", 
    "自營商", "大盤", "殖利率", "供應鏈", "目標價", "漲停", "跌停"
]
# 加入財經專有名詞
for word in custom_finance_words:
    jieba.add_word(word)
    
# 自動將 DataFrame/Dictionary 裡的所有「台股公司簡稱」加入 Jieba，避免被切碎
for stock_name in VALID_STOCK_DICT.values():
    jieba.add_word(stock_name)

def validate_and_filter_stocks(raw_stocks_list):
    verified_stocks = []
    if not isinstance(raw_stocks_list, list): 
        return verified_stocks
    for item in raw_stocks_list:
        code = str(item).split()[0] if " " in str(item) else str(item)
        if code in VALID_STOCK_DICT:
            verified_stocks.append(f"{code} {VALID_STOCK_DICT[code]}")
    return verified_stocks

# ---------------------------------------------------------
# 💡 核心優化 B：暴力擴大爬蟲範圍 (解決內文過短/爬取失敗)
# ---------------------------------------------------------
def fetch_and_extract_keywords(news_id: str) -> str:
    url = f"https://news.cnyes.com/news/id/{news_id}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    
    try:
        res = requests.get(url, headers=headers, timeout=8)
        if res.status_code != 200: 
            return ""
        
        soup = BeautifulSoup(res.text, "html.parser")
        
        # 暴力洗淨：強制拔除所有圖片、影片、廣告腳本，只留純文字
        for tag in soup(["script", "style", "img", "video", "iframe", "figure"]):
            tag.decompose()
            
        # 擴展選擇器：針對 VIP、影音版、特殊排版層層遞進，最後用 find_all("p") 兜底
        paragraphs = (soup.select("section.article-body p") or 
                      soup.select("div[itemprop='articleBody'] p") or
                      soup.select("div.news-content p") or
                      soup.find_all("p"))
                      
        raw_text = " ".join([p.text.strip() for p in paragraphs if len(p.text.strip()) > 5])
        
        if len(raw_text) < 50:
            return ""
            
        # 使用剛才吃過大補丸的 Jieba，精準萃取 Top 30 關鍵字
        keywords = jieba.analyse.extract_tags(raw_text, topK=30, withWeight=False)
        return "、".join(keywords)
        
    except Exception as e:
        print(f"    [爬蟲異常] {e}")
        return ""

# ---------------------------------------------------------
# 💡 核心優化 C：放寬 AI 逾時限制 (解決 504 Deadline expired)
# ---------------------------------------------------------
def analyze_sentiment_and_entities(news_title: str, keywords_str: str) -> dict:
    if not keywords_str:
        return {"sentiment_score": 0.0, "affected_stocks": [], "reasoning": "內文過短或純影音，無文字可供分析"}

    prompt = f"""
    任務：台股新聞量化萃取。
    【標題】："{news_title}"
    【內文關鍵字】："{keywords_str}"

    提取規則：
    1. sentiment_score：-1.0(極空) 到 1.0(極多)。中立或大盤請給 0.0。
    2. affected_stocks：只提取受衝擊台股，格式 ["代號 名稱"]。無明確個股則給 []。
    3. reasoning：20字以內客觀事實。
    請只回傳合法 JSON 格式。
    """
    try:
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.0, 
                response_mime_type="application/json"
            ),
            # 將等待時間從 10 秒放寬至 30 秒，容忍 Google 免費伺服器的尖峰延遲
            request_options={"timeout": 30} 
        )
        return json.loads(response.text)
    except Exception as e:
        return {"sentiment_score": 0.0, "affected_stocks": [], "reasoning": f"解析異常: {e}"}

def get_taiwan_news_optimized(limit=10):
    urls = [
        "https://api.cnyes.com/media/api/v1/newslist/category/tw_stock",
        "https://api.cnyes.com/media/api/v1/newslist/category/headline"
    ]
    headers = {"User-Agent": "Mozilla/5.0"}
    columns = ["publish_time", "affected_stocks", "sentiment_score", "reasoning", "title"]
    records = []
    seen_ids = set()

    print(f"⏳ 啟動終極 NLP 降維萃取管線 (預計處理 {limit} 筆)...")
    
    for url in urls:
        try:
            res = requests.get(url, headers=headers, params={"limit": limit}, timeout=8)
            if res.status_code != 200: 
                continue
                
            for it in res.json().get("items", {}).get("data", []):
                if len(records) >= limit: 
                    break
                    
                news_id = it.get("newsId")
                title = it.get("title", "")
                if not news_id or news_id in seen_ids: 
                    continue
                seen_ids.add(news_id)
                
                print(f"  👉 處理進度 [{len(records)+1}/{limit}]: {title[:20]}...")
                pub_time = datetime.fromtimestamp(it.get("publishAt", 0)).strftime("%m-%d %H:%M")
                
                # 執行淨化爬蟲與 Jieba 壓縮
                keywords_str = fetch_and_extract_keywords(news_id)
                
                # 送入 Gemini 評分 (享有 30 秒寬限期)
                llm_result = analyze_sentiment_and_entities(title, keywords_str)
                filtered_stocks = validate_and_filter_stocks(llm_result.get("affected_stocks", []))
                
                records.append({
                    "publish_time": pub_time,
                    "affected_stocks": ", ".join(filtered_stocks),
                    "sentiment_score": llm_result.get("sentiment_score", 0.0),
                    "reasoning": llm_result.get("reasoning", ""),
                    "title": title
                })
                
                time.sleep(2) # 防超速 429 暫停
                
        except Exception as e:
            print(f"❌ 迴圈層級異常: {e}")
            
        if len(records) >= limit: 
            break

    return pd.DataFrame(records, columns=columns)

# ==========================================
# 執行與輸出
# ==========================================
df_news = get_taiwan_news_optimized(limit=10)
print(f"\n✅ Cell 3: 成功提取 {len(df_news)} 筆")
if not df_news.empty:
    display(df_news[["publish_time", "affected_stocks", "sentiment_score", "reasoning", "title"]])

⏳ 啟動終極 NLP 降維萃取管線 (預計處理 10 筆)...
  👉 處理進度 [1/10]: 中鋼9月稅前盈餘月減19% 前三季賺43...
  👉 處理進度 [2/10]: 〈央行報告搶先看〉楊金龍：維持偏緊貨幣政...
  👉 處理進度 [3/10]: 遠傳攜手Elveo Mobile 推動手...
  👉 處理進度 [4/10]: 〈央行報告搶先看〉楊金龍示警二種狀況傳遞...
  👉 處理進度 [5/10]: 台股近兆巨量震盪，非理性錯殺浮現甜甜價？...
  👉 處理進度 [6/10]: 〈力成法說〉漲價效益延續帶動Q4獲利攀升...
  👉 處理進度 [7/10]: 〈力成法說〉Q3 EPS 3.55元創新...
  👉 處理進度 [8/10]: 聯一光9月自結EPS0.12元虧轉盈 前...
  👉 處理進度 [9/10]: 國泰世華銀布局企業穩定幣收款與結算服務 ...
  👉 處理進度 [10/10]: 華碩看旺2027年台灣商用PC市場 迎量...

✅ Cell 3: 成功提取 10 筆


,publish_time,affected_stocks,sentiment_score,reasoning,title
0,10-27 16:43,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",中鋼9月稅前盈餘月減19% 前三季賺43億元
1,10-27 16:39,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",〈央行報告搶先看〉楊金龍：維持偏緊貨幣政策 信用資源流向實質生產
2,10-27 16:38,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",遠傳攜手Elveo Mobile 推動手機直連衛星技術與服務
3,10-27 16:38,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",〈央行報告搶先看〉楊金龍示警二種狀況傳遞效果 恐放大金融市場波動
4,10-27 16:37,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",台股近兆巨量震盪，非理性錯殺浮現甜甜價？搶反彈先避開致命死穴
5,10-27 16:33,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",〈力成法說〉漲價效益延續帶動Q4獲利攀升 盼與晶圓廠策略聯盟
6,10-27 16:26,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",〈力成法說〉Q3 EPS 3.55元創新高 Q4營收毛利率、稼動率可望續揚
7,10-27 16:21,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",聯一光9月自結EPS0.12元虧轉盈 前三季每股約賺1.82元
8,10-27 15:40,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",國泰世華銀布局企業穩定幣收款與結算服務 串聯傳統金融與虛擬資產生態系
9,10-27 15:31,,0.0,"解析異常: 429 You exceeded your current quota, please check your plan and billin...",華碩看旺2027年台灣商用PC市場 迎量價齊揚與標案預算回溫


In [28]:
import os
from datetime import datetime

# 定義儲存路徑與建立資料夾
base_path = r"C:\Users\JDA\Desktop\kalshi 台灣股市預測\news_sentiment_data"
os.makedirs(base_path, exist_ok=True)

# 執行存檔
if 'df_news' in locals() and not df_news.empty:
    file_name = f"{base_path}\\news_sentiment_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df_news.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"📁 台灣新聞與 AI 情緒分數已獨立儲存至: {file_name}")

📁 台灣新聞與 AI 情緒分數已獨立儲存至: C:\Users\JDA\Desktop\kalshi 台灣股市預測\news_sentiment_data\news_sentiment_20261027_1644.csv


# 即時台股

In [29]:
# ==========================================
# 📊 Cell 4：即時台股技術面與價量爬蟲 (獨立模組)
# ==========================================
import yfinance as yf
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

def fetch_taiwan_stock_technicals(watchlist):
    """
    抓取自選股的即時價量與技術指標 (RSI, MACD, 布林通道)
    """
    print("⏳ 正在即時爬取台股技術面特徵...")
    tech_records = []
    
    for item in watchlist:
        stock_code = item["code"]
        stock_name = item["name"]
        ticker = f"{stock_code}.TW"
        
        try:
            df = yf.Ticker(ticker).history(period="3mo")
            if df.empty:
                continue
                
            # 計算布林通道
            df['SMA20'] = df['Close'].rolling(window=20).mean()
            df['STD20'] = df['Close'].rolling(window=20).std()
            df['Upper_BB'] = df['SMA20'] + (df['STD20'] * 2)
            df['Lower_BB'] = df['SMA20'] - (df['STD20'] * 2)
            
            # 計算 RSI (14日)
            delta = df['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            df['RSI'] = 100 - (100 / (1 + (gain / loss)))
            
            # 計算 MACD (判斷中期趨勢)
            exp1 = df['Close'].ewm(span=12, adjust=False).mean()
            exp2 = df['Close'].ewm(span=26, adjust=False).mean()
            df['MACD'] = exp1 - exp2
            df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
            df['MACD_Hist'] = df['MACD'] - df['Signal_Line']
            
            latest = df.iloc[-1]
            
            tech_records.append({
                "code": stock_code,
                "name": stock_name,
                "close": latest['Close'],
                "rsi": latest['RSI'],
                "upper_bb": latest['Upper_BB'],
                "lower_bb": latest['Lower_BB'],
                "macd_hist": latest['MACD_Hist'] # 大於0為多頭，小於0為空頭
            })
            print(f"  ✅ {stock_name} ({stock_code}) 技術指標爬取完成")
            
        except Exception as e:
            print(f"  ❌ {stock_name} 爬取失敗: {e}")
            
    return pd.DataFrame(tech_records)

# 執行 Cell 4
watchlist_targets = [
    {"name": "台積電", "code": "2330", "kalshi_keywords": ["semiconductor", "Taiwan", "NVDA", "Nvidia", "Fed", "rate", "TSMC"]},
    {"name": "鴻海", "code": "2317", "kalshi_keywords": ["Apple", "iPhone", "tariff", "supply", "China"]},
    {"name": "聯發科", "code": "2454", "kalshi_keywords": ["semiconductor", "Tech", "Nasdaq", "Inflation"]}
]

df_tech = fetch_taiwan_stock_technicals(watchlist_targets)
display(df_tech)

⏳ 正在即時爬取台股技術面特徵...
  ✅ 台積電 (2330) 技術指標爬取完成
  ✅ 鴻海 (2317) 技術指標爬取完成
  ✅ 聯發科 (2454) 技術指標爬取完成


,code,name,close,rsi,upper_bb,lower_bb,macd_hist
0,2330,台積電,2400.0,15.686275,2608.970454,2344.029546,-16.662011
1,2317,鴻海,247.5,43.689320,257.719119,237.080881,0.190553
2,2454,聯發科,3960.0,6.181818,5403.878146,3713.121854,-95.481147


In [30]:
import os
from datetime import datetime

# 定義儲存路徑與建立資料夾
base_path = r"C:\Users\JDA\Desktop\kalshi 台灣股市預測\stock_technicals_data"
os.makedirs(base_path, exist_ok=True)

# 執行存檔
if 'df_tech' in locals() and not df_tech.empty:
    # 加入當天日期戳記方便後續合併
    df_tech_save = df_tech.copy()
    df_tech_save['trade_date'] = datetime.now().strftime('%Y-%m-%d')
    
    file_name = f"{base_path}\\technicals_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df_tech_save.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"📁 即時技術面與價量已獨立儲存至: {file_name}")

📁 即時技術面與價量已獨立儲存至: C:\Users\JDA\Desktop\kalshi 台灣股市預測\stock_technicals_data\technicals_20261027_1644.csv


# 預警

In [31]:
# ==========================================
# 🔮 Cell 5：台股五日趨勢預警系統 (結合 AI 與宏觀)
# ==========================================
from datetime import datetime

def generate_5_day_forecast(watchlist, df_n, df_k, df_t):
    """
    根據新聞情緒(短效)、MACD(中效)與宏觀/布林通道(長效)，推算未來 5 天逐日路徑
    """
    print("=" * 70)
    print("🚀 啟動台股五日趨勢預警系統")
    print("=" * 70)
    
    # 1. 計算 Kalshi 總經大盤基底分數 (Cell 2 資料)
    global_macro_score = 0.0
    if not df_k.empty:
        core_keywords = ["Fed", "rate", "inflation", "CPI", "S&P", "Nasdaq", "semiconductor", "Taiwan", "NVDA"]
        combined_titles = df_k["event_title"].fillna("") + " " + df_k["market_title"].fillna("")
        matches = df_k[combined_titles.str.contains("|".join(core_keywords), case=False, na=False)]
        if not matches.empty and matches["volume"].sum() > 0:
            weighted_prob = (matches["implied_prob"] * matches["volume"]).sum() / matches["volume"].sum()
            global_macro_score = weighted_prob - 0.5

    for item in watchlist:
        stock_code = item["code"]
        stock_name = item["name"]
        
        # 2. 獲取 Cell 3 該股的最新 AI 新聞情緒
        sentiment = 0.0
        if not df_n.empty and "affected_stocks" in df_n.columns:
            matched_news = df_n[df_n["affected_stocks"].str.contains(stock_code, case=False, na=False)]
            if not matched_news.empty:
                sentiment = matched_news["sentiment_score"].mean()
                
        # 3. 獲取 Cell 4 該股的技術面
        tech_row = df_t[df_t["code"] == stock_code]
        if tech_row.empty:
            continue
            
        rsi = tech_row.iloc[0]["rsi"]
        macd_hist = tech_row.iloc[0]["macd_hist"]
        close_price = tech_row.iloc[0]["close"]
        upper_bb = tech_row.iloc[0]["upper_bb"]
        lower_bb = tech_row.iloc[0]["lower_bb"]
        
        print(f"\n🎯 【{stock_code} {stock_name}】次週預警報告")
        print(f"   (AI 情緒: {sentiment:+.2f} | RSI: {rsi:.1f} | MACD 動能: {macd_hist:+.2f} | 總經大盤: {global_macro_score:+.2f})")
        print("-" * 50)
        
        # 4. 逐日推演邏輯
        days = ["週一 (T+1)", "週二 (T+2)", "週三 (T+3)", "週四 (T+4)", "週五 (T+5)"]
        for i, day in enumerate(days):
            day_idx = i + 1
            
            # T+1, T+2 受新聞情緒影響最大；T+4, T+5 新聞效應歸零，完全看大盤與均值回歸
            news_weight = max(0, 0.8 - (day_idx * 0.2)) 
            macro_weight = min(0.6, 0.2 + (day_idx * 0.1))
            
            # 當日綜合分數
            daily_score = (sentiment * news_weight) + (global_macro_score * macro_weight)
            
            # 加入技術面干預
            if day_idx <= 3 and macd_hist > 0: 
                daily_score += 0.1 # MACD 偏多，支撐前三日
            elif day_idx <= 3 and macd_hist < 0:
                daily_score -= 0.1 # MACD 偏空，壓抑前三日
                
            # 極端值反轉預測 (到了週四、週五，極易觸碰布林通道而均值回歸)
            if rsi >= 75 or close_price >= upper_bb:
                if day_idx >= 3: daily_score -= 0.3 # 預計週三後漲多回檔
            elif rsi <= 25 or close_price <= lower_bb:
                if day_idx >= 3: daily_score += 0.3 # 預計週三後跌深反彈
                
            # 判定燈號
            if daily_score >= 0.15:
                trend = "🔴 收漲 (機率偏高)"
            elif daily_score <= -0.15:
                trend = "🟢 收跌 (機率偏高)"
            else:
                trend = "⚪ 震盪 / 盤整"
                
            print(f"   🗓️ {day} : {trend}  (推算動能: {daily_score:+.2f})")

# 執行 Cell 5 (請確認 df_news 與 df_kalshi 在記憶體中存在)
if 'df_news' not in locals(): df_news = pd.DataFrame()
if 'df_kalshi' not in locals(): df_kalshi = pd.DataFrame()

generate_5_day_forecast(watchlist_targets, df_news, df_kalshi, df_tech)

🚀 啟動台股五日趨勢預警系統

🎯 【2330 台積電】次週預警報告
   (AI 情緒: +0.00 | RSI: 15.7 | MACD 動能: -16.66 | 總經大盤: +0.00)
--------------------------------------------------
   🗓️ 週一 (T+1) : ⚪ 震盪 / 盤整  (推算動能: -0.10)
   🗓️ 週二 (T+2) : ⚪ 震盪 / 盤整  (推算動能: -0.10)
   🗓️ 週三 (T+3) : 🔴 收漲 (機率偏高)  (推算動能: +0.20)
   🗓️ 週四 (T+4) : 🔴 收漲 (機率偏高)  (推算動能: +0.30)
   🗓️ 週五 (T+5) : 🔴 收漲 (機率偏高)  (推算動能: +0.30)

🎯 【2317 鴻海】次週預警報告
   (AI 情緒: +0.00 | RSI: 43.7 | MACD 動能: +0.19 | 總經大盤: +0.00)
--------------------------------------------------
   🗓️ 週一 (T+1) : ⚪ 震盪 / 盤整  (推算動能: +0.10)
   🗓️ 週二 (T+2) : ⚪ 震盪 / 盤整  (推算動能: +0.10)
   🗓️ 週三 (T+3) : ⚪ 震盪 / 盤整  (推算動能: +0.10)
   🗓️ 週四 (T+4) : ⚪ 震盪 / 盤整  (推算動能: +0.00)
   🗓️ 週五 (T+5) : ⚪ 震盪 / 盤整  (推算動能: +0.00)

🎯 【2454 聯發科】次週預警報告
   (AI 情緒: +0.00 | RSI: 6.2 | MACD 動能: -95.48 | 總經大盤: +0.00)
--------------------------------------------------
   🗓️ 週一 (T+1) : ⚪ 震盪 / 盤整  (推算動能: -0.10)
   🗓️ 週二 (T+2) : ⚪ 震盪 / 盤整  (推算動能: -0.10)
   🗓️ 週三 (T+3) : 🔴 收漲 (機率偏高)  (推算動能: +0.20)
   🗓️ 週四 (T+4) : 🔴 收漲 (機率偏高)  (推算動能

In [32]:
# ==========================================
# 📊 Cell 7：過去一個月歷史回測與勝率驗證引擎
# ==========================================

import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

def run_historical_backtest(stock_code="2330", backtest_days=20):
    print(f"⏳ 正在啟動【{stock_code}】歷史回測 (回測窗格: 過去 {backtest_days} 個交易日)...")
    
    # 1. 抓取過去 3 個月資料 (保留足夠天數計算 20MA 與 RSI)
    ticker = f"{stock_code}.TW"
    df = yf.Ticker(ticker).history(period="3mo")
    
    if df.empty or len(df) < backtest_days + 30:
        print("資料不足以進行回測")
        return

    # 2. 批次計算歷史技術指標
    df['SMA20'] = df['Close'].rolling(window=20).mean()
    df['STD20'] = df['Close'].rolling(window=20).std()
    df['Upper_BB'] = df['SMA20'] + (df['STD20'] * 2)
    df['Lower_BB'] = df['SMA20'] - (df['STD20'] * 2)
    
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / loss)))
    
    exp1 = df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = exp1 - exp2
    df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['Signal_Line']
    
    df = df.dropna()

    # 3. 定義回測計分板
    # T+1 到 T+5 的 [預測正確次數, 總預測次數 (排除中立)]
    accuracy_tracker = {1: [0, 0], 2: [0, 0], 3: [0, 0], 4: [0, 0], 5: [0, 0]}
    
    # 擷取回測區間 (扣除最後 5 天，因為最後 5 天沒有「未來 5 天」的答案可對答案)
    end_idx = len(df) - 5
    start_idx = end_idx - backtest_days
    
    print("-" * 60)
    
    # 4. 執行滑動窗格回測
    for i in range(start_idx, end_idx):
        current_date = df.index[i].strftime('%Y-%m-%d')
        row = df.iloc[i]
        
        # 模擬特徵 (實戰時從 DB 撈取當日的真實 AI 與 Kalshi 分數)
        sentiment = 0.0 
        global_macro_score = 0.0
        
        rsi = row['RSI']
        macd_hist = row['MACD_Hist']
        close_price = row['Close']
        upper_bb = row['Upper_BB']
        lower_bb = row['Lower_BB']
        
        # 提取未來 5 天的真實收盤價來「對答案」
        future_closes = df['Close'].iloc[i+1 : i+6].values
        
        for day_idx in range(1, 6):
            # 複製 Cell 5 的逐日推演邏輯
            news_weight = max(0, 0.8 - (day_idx * 0.2))
            macro_weight = min(0.6, 0.2 + (day_idx * 0.1))
            daily_score = (sentiment * news_weight) + (global_macro_score * macro_weight)
            
            if day_idx <= 3 and macd_hist > 0: daily_score += 0.1
            elif day_idx <= 3 and macd_hist < 0: daily_score -= 0.1
                
            if rsi >= 75 or close_price >= upper_bb:
                if day_idx >= 3: daily_score -= 0.3
            elif rsi <= 25 or close_price <= lower_bb:
                if day_idx >= 3: daily_score += 0.3
                
            # 判斷預測方向 (1: 漲, -1: 跌, 0: 中立)
            if daily_score >= 0.15: pred_dir = 1
            elif daily_score <= -0.15: pred_dir = -1
            else: pred_dir = 0
            
            # 判斷真實方向 (對比當日收盤價)
            actual_dir = 1 if future_closes[day_idx-1] > close_price else -1
            
            # 紀錄勝率 (忽略中立訊號)
            if pred_dir != 0:
                accuracy_tracker[day_idx][1] += 1 # 總出手次數 +1
                if pred_dir == actual_dir:
                    accuracy_tracker[day_idx][0] += 1 # 預測正確 +1

    # 5. 印出回測勝率報表
    print(f"📈 【{stock_code}】過去 {backtest_days} 日盲測勝率報告 (純技術面邏輯)")
    print("-" * 60)
    for day_idx in range(1, 6):
        hits, total = accuracy_tracker[day_idx]
        if total > 0:
            win_rate = (hits / total) * 100
            print(f"   • T+{day_idx} 預測勝率: {win_rate:5.1f}%  (命中 {hits}/{total} 次)")
        else:
            print(f"   • T+{day_idx} 預測勝率: 無明確訊號 (皆為中立盤整)")
    print("=" * 60)

# 執行回測 (以台積電為例，回測過去 20 個交易日)
run_historical_backtest(stock_code="2330", backtest_days=20)

⏳ 正在啟動【2330】歷史回測 (回測窗格: 過去 20 個交易日)...
------------------------------------------------------------
📈 【2330】過去 20 日盲測勝率報告 (純技術面邏輯)
------------------------------------------------------------
   • T+1 預測勝率: 無明確訊號 (皆為中立盤整)
   • T+2 預測勝率: 無明確訊號 (皆為中立盤整)
   • T+3 預測勝率: 100.0%  (命中 3/3 次)
   • T+4 預測勝率: 100.0%  (命中 3/3 次)
   • T+5 預測勝率: 100.0%  (命中 3/3 次)


# 預測

In [33]:
# ==========================================
# 📈 Cell 8：XGBoost 明日具體價格預測引擎 (絕對數值版)
# ==========================================

import yfinance as yf
import pandas as pd
import xgboost as xgb
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

def predict_tomorrow_exact_price(stock_code="2330", stock_name="台積電"):
    print(f"⏳ 啟動【{stock_code} {stock_name}】純技術面價格預測模型...")
    
    # 1. 準備真實歷史資料
    ticker = f"{stock_code}.TW"
    df = yf.Ticker(ticker).history(period="1y")
    
    if df.empty: 
        return
        
    # 2. 真實技術特徵工程 (無任何隨機亂數)
    df['SMA20'] = df['Close'].rolling(window=20).mean()
    df['STD20'] = df['Close'].rolling(window=20).std()
    
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / loss)))
    
    # 3. 定義預測標籤「次日真實報酬率」
    df['Target_Return'] = (df['Close'].shift(-1) - df['Close']) / df['Close']
    
    feature_cols = ['RSI', 'SMA20', 'STD20', 'Volume']
    
    # 4. 嚴格對齊今日與歷史特徵
    X_latest = df[feature_cols].iloc[-1:]
    current_price = df['Close'].iloc[-1]
    
    df_train = df.iloc[:-1].dropna()
    X_train = df_train[feature_cols]
    y_train = df_train['Target_Return']
    
    # 5. 建立與訓練模型 (鎖定 Random State 確保每次結果一致)
    model_reg = xgb.XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        objective='reg:squarederror',
        random_state=42
    )
    
    model_reg.fit(X_train, y_train)
    
    # 6. 推算明日價格
    predicted_return = model_reg.predict(X_latest)[0]
    predicted_price = current_price * (1 + predicted_return)
    
    daily_volatility = df_train['Target_Return'].std()
    upper_bound = current_price * (1 + predicted_return + daily_volatility)
    lower_bound = current_price * (1 + predicted_return - daily_volatility)

    # 處理時間邏輯
    now = datetime.now()
    tomorrow = now + timedelta(days=1)
    
    print("=" * 60)
    print(f"🎯 【{stock_name}】明日價格預測報告")
    print("=" * 60)
    print(f"   • 基準日收盤價 : {current_price:.2f} 元 ({now.strftime('%Y-%m-%d')})")
    print(f"   • 預測標的日期 : {tomorrow.strftime('%Y-%m-%d')} (下一個交易日)")
    print("-" * 60)
    print(f"   ➤ 模型預估報酬 : {predicted_return * 100:+.2f}%")
    print(f"   ➤ 預測核心收盤 : {predicted_price:.2f} 元")
    print(f"   ➤ 預期震盪區間 : {lower_bound:.2f} 元 ~ {upper_bound:.2f} 元")
    print("=" * 60)

# 執行測試
predict_tomorrow_exact_price("2330", "台積電")

⏳ 啟動【2330 台積電】純技術面價格預測模型...
🎯 【台積電】明日價格預測報告
   • 基準日收盤價 : 2400.00 元 (2026-10-27)
   • 預測標的日期 : 2026-10-28 (下一個交易日)
------------------------------------------------------------
   ➤ 模型預估報酬 : +3.09%
   ➤ 預測核心收盤 : 2474.18 元
   ➤ 預期震盪區間 : 2425.69 元 ~ 2522.67 元


In [38]:
import os
import pandas as pd
from datetime import datetime

# 執行預測並取得變數 (假設你已經執行了原本的預測函數，我們將結果轉成 DataFrame)
# 在實戰中，你可以將預測函數改寫為 return 這些值，這裡直接將所有自選股的結果存檔
base_path = r"C:\Users\JDA\Desktop\kalshi 台灣股市預測\forecast_data"
os.makedirs(base_path, exist_ok=True)

forecast_records = []
now = datetime.now()

# 針對你的 watchlist 批次執行明日價格預測並紀錄
for item in [{"name": "台積電", "code": "2330"}, {"name": "鴻海", "code": "2317"}, {"name": "聯發科", "code": "2454"}]:
    # 這裡呼叫你寫好的 XGBoost 預測函數 (為確保能接住回傳值，建議函數最後 return 一個 dict)
    # 以下為模擬函數回傳封裝，實際依你的推論結果為準
    try:
        predict_tomorrow_exact_price(item["code"], item["name"])
        # 若需要精準存入 CSV，請將 predict_tomorrow_exact_price 的 print 改為 return dict
        # forecast_records.append(result_dict)
    except Exception as e:
        print(f"❌ {item['name']} 預測失敗: {e}")

# 若函數已改為 return dict，即可執行下方程式碼將結果轉為 CSV 存檔
# if forecast_records:
#     df_forecast = pd.DataFrame(forecast_records)
#     file_name = f"{base_path}\\price_forecast_{now.strftime('%Y%m%d')}.csv"
#     df_forecast.to_csv(file_name, index=False, encoding='utf-8-sig')
#     print(f"📁 明日價格預測總表已儲存至: {file_name}")

⏳ 啟動【2330 台積電】純技術面價格預測模型...
🎯 【台積電】明日價格預測報告
   • 基準日收盤價 : 2400.00 元 (2026-10-27)
   • 預測標的日期 : 2026-10-28 (下一個交易日)
------------------------------------------------------------
   ➤ 模型預估報酬 : +3.09%
   ➤ 預測核心收盤 : 2474.18 元
   ➤ 預期震盪區間 : 2425.69 元 ~ 2522.67 元
⏳ 啟動【2317 鴻海】純技術面價格預測模型...
🎯 【鴻海】明日價格預測報告
   • 基準日收盤價 : 247.50 元 (2026-10-27)
   • 預測標的日期 : 2026-10-28 (下一個交易日)
------------------------------------------------------------
   ➤ 模型預估報酬 : +0.71%
   ➤ 預測核心收盤 : 249.25 元
   ➤ 預期震盪區間 : 243.50 元 ~ 254.99 元
⏳ 啟動【2454 聯發科】純技術面價格預測模型...
🎯 【聯發科】明日價格預測報告
   • 基準日收盤價 : 3960.00 元 (2026-10-27)
   • 預測標的日期 : 2026-10-28 (下一個交易日)
------------------------------------------------------------
   ➤ 模型預估報酬 : +1.35%
   ➤ 預測核心收盤 : 4013.53 元
   ➤ 預期震盪區間 : 3850.23 元 ~ 4176.84 元
